<a href="https://colab.research.google.com/github/vwendtUDC/Knee_or_ROC/blob/main/Compact_CCT_multiOb_val.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install tqdm

In [2]:
%matplotlib inline

import torch
from torchvision import datasets
from torchvision.transforms import ToTensor
from torchvision.transforms import ToPILImage
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

In [3]:
#Converts a tensor image to the proper numpy format of (x, y, channels) from a format of (channels, x, y).
def im_transform(tensor):
    if tensor.shape[0]==3 or tensor.shape[0]==1:
      return tensor.permute(1,2,0).numpy()
    else:
      print('It seems this image is already formatted properly for numpy operations.')

#The concatenating image function for combining four images with four labels into a single grid-like image.
#For this the indices of images and the dataset they are from must be defined. It is optional to pull out
#The corresponding labels or to resize to other dimensions.
def concat_image(indices,dataset,labels=False,resizing=None):
    cc_img=np.zeros((64,64,3))
    cats=[]
    for i in range(4):
        temp_img=im_transform(dataset[indices[i]][0])
        cats.append(dataset[indices[i]][1])
        if i == 0:
            cc_img[0:32,0:32]=temp_img
        elif i == 1:
            cc_img[0:32,32:64]=temp_img
        elif i == 2:
            cc_img[32:64,0:32]=temp_img
        else:
            cc_img[32:64,32:64]=temp_img
    if resizing:
        from skimage.transform import resize
        try:
            cc_img=resize(cc_img,resizing)
        except:
            print('Make sure the resizing input is given as a tuple of the desired dimensions')
    if labels:
        return cc_img,tuple(cats)
    else:
        return cc_img

#Converts an image stored as a numpy array back to the proper format for saving the image through the PIL package.
def revert_np_image(im_array):
  tmp=ToTensor()(im_array)
  return ToPILImage()(tmp)

In [4]:
#This function generates random integers to use as indices for pulling images from the original database for concatenation.
#Inputs here are i_max, which is the total number of images in the original database,
#and iterations, which is the number of new images one plans to generate.
def rand_indices(i_max,iterations):
    from random import randint
    i_max=i_max-1
    indices=[]
    for i in range(iterations):
        i1=randint(0,i_max)
        i2=randint(0,i_max)
        while i2 == i1:
            i2=randint(0,i_max)
        i3=randint(0,i_max)
        while i3 == i2 or i3 == i1:
            i3=randint(0,i_max)
        i4=randint(0,i_max)
        while i4 == i3 or i4 == i2 or i4 == i1:
            i4=randint(0,i_max)
        indices.append(tuple([i1,i2,i3,i4]))
    return indices


#This function simply creates a new destination folder to dump the concatenated images.
#By default, this function will create a folder in the current directory unless otherwise specified.
def set_directories(directory=None):
  if not directory:
    directory='/temp_dir/'
  if directory[0]!='/':
    return print('Make sure you have a new folder properly formatted with a \"/\" at the beginning')
  else:
    import os
    cd=os.getcwd()
    target=cd+directory
    if not os.path.exists(target):
      os.makedirs(target)
    return target

#This function initializes a dataframe formatted to track the original indices
#that are randomized to specific concatenation of the original  pulled, their respective labels,
#and the new filenames saved within the new database.
def create_dataframe(num_entries,base_data):
  import pandas as pd
  import numpy as np
  dataframe=pd.DataFrame()
  dataframe['Indices']=rand_indices(len(base_data),num_entries)
  dataframe['Label_1']=np.zeros(num_entries,dtype=object)
  dataframe['Label_2']=np.zeros(num_entries,dtype=object)
  dataframe['Label_3']=np.zeros(num_entries,dtype=object)
  dataframe['Label_4']=np.zeros(num_entries,dtype=object)
  dataframe['Filename']=np.zeros(num_entries,dtype=object)
  return dataframe

#This function generates the entire database through the use of the prior functions.
#Only the size of the data base through num_entries and the original dataset need to be provided,
#and everything else will be initialized to a default value. It's recommended to input the image_dest
#and df_name as specific file destinations/final table name if generating multiple datasets.
def generate_data(num_entries,base_data,image_dest=None,resizing=None,dataframe=None,df_name=None):
  import os
  if not image_dest:
    image_dest=set_directories()
  else:
    image_dest=set_directories(image_dest)
  if not resizing:
    resizing=(32,32)
  if not dataframe:
    dataframe=create_dataframe(num_entries,base_data)
  if not df_name:
    df_name='concated_data.csv'

  from tqdm import tqdm

  for i in tqdm(range (num_entries),desc='Generating ... '):
    temp_ind=dataframe['Indices'][i]
    temp_im,labels=concat_image(temp_ind,base_data,labels=True,resizing=resizing)
    if num_entries > 100000:
      name=image_dest+f'{i+1:09d}'+'.jpg'
    else:
      name=image_dest+f'{i+1:06d}'+'.jpg'
    revert_np_image(temp_im).save(name)
    dataframe.at[i,'Filename']=name
    dataframe.at[i,'Label_1']=labels[0]
    dataframe.at[i,'Label_2']=labels[1]
    dataframe.at[i,'Label_3']=labels[2]
    dataframe.at[i,'Label_4']=labels[3]
    dataframe.to_csv(os.getcwd()+'/'+df_name)
  print('\nThe data table is saved to '+df_name+', and images are stored within '+image_dest+'.')
  return dataframe

In [5]:
def pic_label_show(image=None,labels=None,dataframe=None,predictions=False):
    if type(image)!=torch.Tensor:
      from PIL import Image
      if not labels:
        try:
          labels=dataframe.loc[dataframe['Filename']==image,'Labels'][1]
        except:
          return print('The image path could not be located. Operation has been aborted.')
      try:
        image=Image.open(image)
        image=ToTensor()(image)
      except:
        return print('The image input is formatted incorrectly. Either load in a tensor corresponding to the image or the correct file path.')
    cat_keys={
     0:'airplane',
    1:'automobile',
    2:'bird',
    3:'cat',
    4:'deer',
    5:'dog',
    6:'frog',
    7:'horse',
    8:'ship',
    9:'truck'
    }
    if image.shape[0]==3 or image.shape[0]==1:
      image=im_transform(image)
    x=image.shape[0]
    y=image.shape[1]
    fig, ax = plt.subplots()
    if len(labels)==1:
      label=cat_keys[labels]
      try:
        ax.imshow(image)
      except:
        return print("The image format may have been incorrect. Please make sure it is in the numpy image format of (x, y, channels)")
      plt.text(2,2,label,bbox=dict(facecolor='white'))
    else:
        if len(labels) % 4 != 0:
                print('Image does not appear square with four distinct quadrants.')
        elif not predictions:
          cats=[cat_keys[label] for label in labels]
          try:
            ax.imshow(image)
          except:
            return print("The image format may have been incorrect. Please make sure it is in the numpy image format of (x, y, channels)")
        for i in range(len(cats)):
          if i == 0:
            plt.text(2,2,cats[i],bbox=dict(facecolor='white'))
          elif i == 1:
            plt.text(2+int(x/2),2,cats[i],bbox=dict(facecolor='white'))
          elif i == 2:
            plt.text(2,2+int(y/2),cats[i],bbox=dict(facecolor='white'))
          else:
            plt.text(2+int(x/2),2+int(y/2),cats[i],bbox=dict(facecolor='white'))
    return None

In [6]:
# This cell code changed from FashionMNIST to CIFAR10

# Download training data from open datasets.
training_data = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)


100%|██████████| 170M/170M [00:03<00:00, 54.2MB/s]


In [7]:
#using Labsign Data instead.
size_database=500
concat_data=generate_data(size_database,test_data,image_dest='/test/',df_name='test.csv')

Generating ... : 100%|██████████| 500/500 [00:06<00:00, 81.07it/s]


The data table is saved to test.csv, and images are stored within /content/test/.


In [8]:
####above code adds test/test.csv to directory for ease of access when using colab

##Zip and upload utils from https://github.com/SHI-Labs/Compact-Transformers/tree/main/src



In [9]:
%pip install timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━ 424.0/664.8 MB 13.7 MB/s eta 0:00:18ERROR: Operation cancelled by user
   ━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━ 424.0/664.8 MB 13.7 MB/s eta 0:00:18


In [10]:
%pip install ipynb

In [11]:
%pip install kneed

In [12]:
%pip install torch torchvision torchaudio

  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nvjitlink_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
Using cached nvidia_cublas_

In [13]:
!unzip utils.zip

Archive:  utils.zip
   creating: utils/
  inflating: __MACOSX/._utils        
  inflating: utils/transformers.py   
  inflating: __MACOSX/utils/._transformers.py  
 extracting: utils/__init__.py       
  inflating: __MACOSX/utils/.___init__.py  
  inflating: utils/tokenizer.py      
  inflating: __MACOSX/utils/._tokenizer.py  
  inflating: utils/embedder.py       
  inflating: __MACOSX/utils/._embedder.py  
  inflating: utils/stochastic_depth.py  
  inflating: __MACOSX/utils/._stochastic_depth.py  
  inflating: utils/helpers.py        
  inflating: __MACOSX/utils/._helpers.py  


In [14]:
%matplotlib inline

# Original Code from the SHI-Labs
# .utils in the from lines changed to utils
# .registry import changed to the function def (copied from the registry.py)

import torch
from torch.hub import load_state_dict_from_url
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from utils.transformers import TransformerClassifier
from utils.tokenizer import Tokenizer
from utils.helpers import pe_check, fc_check
from torchvision.transforms import ToTensor
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import os
import kneed



In [ ]:
try:
    from timm.models.registry import register_model
except ImportError:
    #from .registry import register_model
    def register_model(func):
      """
      Fallback wrapper in case timm isn't installed
      """
      return func

model_urls = {
    'cct_7_3x1_32':
        'https://shi-labs.com/projects/cct/checkpoints/pretrained/cct_7_3x1_32_cifar10_300epochs.pth'
}


class CCT(nn.Module):
    def __init__(self,
                 img_size=224,
                 embedding_dim=768,
                 n_input_channels=3,
                 n_conv_layers=1,
                 kernel_size=7,
                 stride=2,
                 padding=3,
                 pooling_kernel_size=3,
                 pooling_stride=2,
                 pooling_padding=1,
                 dropout=0.,
                 attention_dropout=0.1,
                 stochastic_depth=0.1,
                 num_layers=14,
                 num_heads=6,
                 mlp_ratio=4.0,
                 num_classes=1000,
                 positional_embedding='learnable',
                 *args, **kwargs):
        super(CCT, self).__init__()

        self.tokenizer = Tokenizer(n_input_channels=n_input_channels,
                                   n_output_channels=embedding_dim,
                                   kernel_size=kernel_size,
                                   stride=stride,
                                   padding=padding,
                                   pooling_kernel_size=pooling_kernel_size,
                                   pooling_stride=pooling_stride,
                                   pooling_padding=pooling_padding,
                                   max_pool=True,
                                   activation=nn.ReLU,
                                   n_conv_layers=n_conv_layers,
                                   conv_bias=False)

        self.classifier = TransformerClassifier(
            sequence_length=self.tokenizer.sequence_length(n_channels=n_input_channels,
                                                           height=img_size,
                                                           width=img_size),
            embedding_dim=embedding_dim,
            seq_pool=True,
            dropout=dropout,
            attention_dropout=attention_dropout,
            stochastic_depth=stochastic_depth,
            num_layers=num_layers,
            num_heads=num_heads,
            mlp_ratio=mlp_ratio,
            num_classes=num_classes,
            positional_embedding=positional_embedding
        )

    def forward(self, x):
        x = self.tokenizer(x)
        return self.classifier(x)


def _cct(arch, pretrained, progress,
         num_layers, num_heads, mlp_ratio, embedding_dim,
         kernel_size=3, stride=None, padding=None,
         positional_embedding='learnable',
         *args, **kwargs):
    stride = stride if stride is not None else max(1, (kernel_size // 2) - 1)
    padding = padding if padding is not None else max(1, (kernel_size // 2))
    model = CCT(num_layers=num_layers,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                embedding_dim=embedding_dim,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                *args, **kwargs)

    if pretrained:
        if arch in model_urls:
            state_dict = load_state_dict_from_url(model_urls[arch],
                                                  progress=progress)
            if positional_embedding == 'learnable':
                state_dict = pe_check(model, state_dict)
            elif positional_embedding == 'sine':
                state_dict['classifier.positional_emb'] = model.state_dict()['classifier.positional_emb']
            state_dict = fc_check(model, state_dict)
            model.load_state_dict(state_dict)
        else:
            raise RuntimeError(f'Variant {arch} does not yet have pretrained weights.')
    return model

def cct_7(arch, pretrained, progress, *args, **kwargs):
    return _cct(arch, pretrained, progress, num_layers=7, num_heads=4, mlp_ratio=2, embedding_dim=256,
                *args, **kwargs)
@register_model
def cct_7_3x1_32(pretrained=False, progress=False,
                 img_size=32, positional_embedding='learnable', num_classes=10,
                 *args, **kwargs):
    return cct_7('cct_7_3x1_32', pretrained, progress,
                 kernel_size=3, n_conv_layers=1,
                 img_size=img_size, positional_embedding=positional_embedding,
                 num_classes=num_classes,
                 *args, **kwargs)




# New Section

##Modified the Workflow from the Quickstart Tutorial of PyTorch


[Learn the Basics](intro.html) ||
**Quickstart** ||
[Tensors](tensorqs_tutorial.html) ||
[Datasets & DataLoaders](data_tutorial.html) ||
[Transforms](transforms_tutorial.html) ||
[Build Model](buildmodel_tutorial.html) ||
[Autograd](autogradqs_tutorial.html) ||
[Optimization](optimization_tutorial.html) ||
[Save & Load Model](saveloadrun_tutorial.html)

# Quickstart
This section runs through the API for common tasks in machine learning. Refer to the links in each section to dive deeper.

## Working with data
PyTorch has two [primitives to work with data](https://pytorch.org/docs/stable/data.html):
``torch.utils.data.DataLoader`` and ``torch.utils.data.Dataset``.
``Dataset`` stores the samples and their corresponding labels, and ``DataLoader`` wraps an iterable around
the ``Dataset``.


PyTorch offers domain-specific libraries such as [TorchText](https://pytorch.org/text/stable/index.html),
[TorchVision](https://pytorch.org/vision/stable/index.html), and [TorchAudio](https://pytorch.org/audio/stable/index.html),
all of which include datasets. For this tutorial, we  will be using a TorchVision dataset.

The ``torchvision.datasets`` module contains ``Dataset`` objects for many real-world vision data like
CIFAR, COCO ([full list here](https://pytorch.org/vision/stable/datasets.html)). In this tutorial, we
use the FashionMNIST dataset. Every TorchVision ``Dataset`` includes two arguments: ``transform`` and
``target_transform`` to modify the samples and labels respectively.



We pass the ``Dataset`` as an argument to ``DataLoader``. This wraps an iterable over our dataset, and supports
automatic batching, sampling, shuffling and multiprocess data loading. Here we define a batch size of 64, i.e. each element
in the dataloader iterable will return a batch of 64 features and labels.



In [ ]:
def im_transform(tensor):
    if tensor.shape[0]==3 or tensor.shape[0]==1:
      return tensor.permute(1,2,0).numpy()
    else:
      print('It seems this image is already formatted properly for numpy operations.')

def pic_label_show(image=None,labels=None,dataframe=None,predictions=False,save=False,file_dest=None):
    if type(image)!=torch.Tensor:
      if not labels:
        try:
          labels=dataframe.loc[dataframe['Filename']==image,'Labels'][1]
        except:
          return print('The image path could not be located. Operation has been aborted.')
      try:
        image=Image.open(image)
        image=ToTensor()(image)
      except:
        return print('The image input is formatted incorrectly. Either load in a tensor corresponding to the image or the correct file path.')
    cat_keys={
    0:'airplane',
    1:'automobile',
    2:'bird',
    3:'cat',
    4:'deer',
    5:'dog',
    6:'frog',
    7:'horse',
    8:'ship',
    9:'truck'
    }
    if image.shape[0]==3 or image.shape[0]==1:
      image=im_transform(image)
    x=image.shape[0]
    y=image.shape[1]
    fig, ax = plt.subplots()
    if len(labels)==1:
      label=cat_keys[labels]
      try:
        ax.imshow(image)
      except:
        return print("The image format may have been incorrect. Please make sure it is in the numpy image format of (x, y, channels)")
      plt.text(2,2,label,bbox=dict(facecolor='white'))
    else:
      if len(labels) % 4 != 0:
        return print('Image does not appear square with four distinct quadrants.')
      else:
        cats=[cat_keys[label] for label in labels]
      try:
        ax.imshow(image)
      except:
        return print("The image format may have been incorrect. Please make sure it is in the numpy image format of (x, y, channels)")
      if not predictions:
        for i in range(len(cats)):
          if i == 0:
            plt.text(2,2,cats[i],bbox=dict(facecolor='white'))
          elif i == 1:
            plt.text(2+int(x/2),2,cats[i],bbox=dict(facecolor='white'))
          elif i == 2:
            plt.text(2,2+int(y/2),cats[i],bbox=dict(facecolor='white'))
          else:
              plt.text(2+int(x/2),2+int(y/2),cats[i],bbox=dict(facecolor='white'))
      else:
        plt.text(0,1,'Possible matches of '+str(cats),bbox=dict(facecolor='white'))
    if save & isinstance(file_dest, str):
      fig.savefig(os.getcwd()+'/'+file_dest+'.png')
    return None


In [ ]:
classes = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]


## Load a model and Predict on a single input

In [ ]:
model=cct_7_3x1_32(pretrained=True)


In [ ]:
print(model)

In [ ]:
def top4_preds(pred):
    list_convert=list(pred[0])
    num_list=[]
    for i in range(len(list_convert)):
        num_list.append(float(list_convert[i]))
    top4=[]
    for i in range(4):
        top4.append(num_list.index(max(num_list)))
        num_list[top4[i]]=-10
    return tuple(top4)

def image_to_pred(file,model):
    im=ToTensor()(Image.open(file))
    with torch.no_grad():
        im=im[None,:]
        pred=model(im)
    return pred

def labels_extract(db,col_names=None):
    if not col_names:
        col_names=['Label_1','Label_2','Label_3','Label_4']
    for i in range(len(col_names)):
        try:
            db[col_names[i]]
        except:
            print('The input database is not formatted correctly to extract labels')
    labels=[]
    for i in range(len(db.index)):
        temp=[]
        for j in range(len(col_names)):
            temp.append(db[col_names[j]][i])
        labels.append(tuple(temp))
    return labels

def images_and_labels(db,label_cols=None,file_col=None):
    if not label_cols:
        label_cols=['Label_1','Label_2','Label_3','Label_4']
    if not file_col:
        file_col='Filename'
    try:
        db[file_col]
    except:
        print('Make sure to input the correct column name for the image file paths.')
    for i in range(len(label_cols)):
        try:
            db[label_cols[i]]
        except:
            print('The input database is not formatted correctly to extract labels')

    files=list(mo_data[file_col])
    labels=labels_extract(db,label_cols)
    return files,labels

def acc_table(labels,predicted,col_names=None):
    if not col_names:
        col_names=['Actuals','Predictions','Num_matches']
    elif len(col_names) != 3:
        return print('This function only creates a table with 3 columns. Make sure only labels for 3 columns are included')
    acc_eval=pd.DataFrame()
    acc_eval[col_names[0]]=labels
    acc_eval[col_names[1]]=predicted
    tot_corr=[]
    for i in range(len(predicted)):
        num_corr=0
        for j in range(len(predicted[i])):
            if predicted[i][j] in labels[i]:
                num_corr+=1
        tot_corr.append(num_corr)
    acc_eval[col_names[2]]=tot_corr
    return acc_eval

def plot_correct_total(tot_corr,db,max_matches=4,match_col='Num_matches',x_title=None,y_title=None,plt_title=None,save=False,file_dest=None):
    num_preds=len(tot_corr)
    if not x_title:
        x_title='Correct matches for given prediction'
    if not y_title:
        y_title='Total number of correct matches'
    if not plt_title:
        plt_title='Distribution of correct matches for '+str(num_preds)+' multi-object predictions'
    fig,ax = plt.subplots()
    corr_cats=list(range(max_matches+1))
    counts=[]
    for i in corr_cats:
        counts.append(sum(db[match_col]==i))
    ax=plt.bar(corr_cats,counts)
    plt.xlabel(x_title)
    plt.ylabel(y_title)
    plt.title(plt_title)
    plt.ylim(ymin=0,ymax=num_preds)
    for rect in ax:
        height = rect.get_height()
        plt.text(rect.get_x() + rect.get_width() / 2.0, height, f'{height:.0f}', ha='center', va='bottom')
    if save & isinstance(file_dest, str):
        fig.savefig(os.getcwd()+'/'+file_dest+'.png')

def plot_correct_prop(tot_corr,db,max_matches=4,match_col='Num_matches',x_title=None,y_title=None,plt_title=None,save=False,file_dest=None):
    num_preds=len(tot_corr)
    if not x_title:
        x_title='Correct matches for given prediction'
    if not y_title:
        y_title='Proportional number of correct matches (%)'
    if not plt_title:
        plt_title='Distribution of correct matches for '+str(num_preds)+' multi-object predictions'
    fig,ax = plt.subplots()
    corr_cats=list(range(max_matches+1))
    counts=[]
    for i in corr_cats:
        counts.append(sum(db[match_col]==i))
    prop_counts = [x / num_preds *100 for x in counts]
    ax=plt.bar(corr_cats,prop_counts)
    plt.xlabel(x_title)
    plt.ylabel(y_title)
    plt.title(plt_title)
    plt.ylim(ymin=0,ymax=100)
    for rect in ax:
        height = rect.get_height()
        plt.text(rect.get_x() + rect.get_width() / 2.0, height, f'{height:.0f} %', ha='center', va='bottom')
    if save & isinstance(file_dest, str):
        fig.savefig(os.getcwd()+'/'+file_dest+'.png')




In [ ]:
def gen_knees(preds,shape=None):
    if shape is None:
        shapes=[]
        for i in range(len(preds)):
            x=range(1,len(preds[i])+1)
            tmp=np.polyfit(x,preds[i],2)
            if tmp[0]<0:
                shapes.append('concave')
            else:
                shapes.append('convex')
    knee_val=[]
    for i in range(len(preds)):
        x=range(1,len(preds[i])+1)
        if shape is not None:
            kneedle=kneed.KneeLocator(x,preds[i],curve=shape,direction='decreasing')
        else:
            kneedle=kneed.KneeLocator(x,preds[i],curve=shapes[i],direction='decreasing')
        knee_val.append(kneedle.knee)
    for i in range(len(knee_val)):
        if knee_val[i] is None:
            knee_val[i]=1
    return knee_val

def plot_knees(preds,knee_val=None,shape=None,x_title=None,y_title=None,plt_title=None,assumption=None,save=False,file_dest=None):
    if not knee_val:
        if not shape:
            knee_val=gen_knees(preds)
        else:
            knee_val=gen_knees(preds,shape=shape)
    num_preds=len(preds)
    if not x_title:
        x_title='# of classes'
    if not y_title:
        y_title='Count of knee values per # of classes'
    if not plt_title:
        plt_title='Distribution of knee values for '+str(num_preds)+' multi-object predictions'
    fig,ax = plt.subplots()
    x=range(1,len(preds[0])+1)
    counts=list(np.zeros(len(preds[0])))
    for i in range(len(knee_val)):
        counts[knee_val[i]-1]+=1
    ax=plt.bar(x,counts)
    plt.xticks(range(1,len(counts)+1))
    plt.xlabel(x_title)
    plt.ylabel(y_title)
    plt.title(plt_title)
    plt.ylim(ymin=0,ymax=1.25*max(counts))
    if not shape:
        plt.annotate('Individually assessed convexity vs concavity',xy=(0.05,0.95),xycoords='axes fraction')
    else:
        plt.annotate('Assumption of all softmax outputs having '+shape+' shapes.',xy=(0.05,0.95),xycoords='axes fraction')
    for rect in ax:
        height = rect.get_height()
        plt.text(rect.get_x() + rect.get_width() / 2.0, height, f'{height:.0f}', ha='center', va='bottom')
    if save & isinstance(file_dest, str):
        fig.savefig(os.getcwd()+'/'+file_dest+'.png')


In [ ]:
#test.csv is created by multiObjectDatasetGen1.ipynb using the labsign data
#for a better understanding of the data and how it is structured read that file as this file mainly deals with the CCT
mo_data=pd.read_csv(os.getcwd()+'/test.csv')

In [ ]:
mo_data.shape

In [ ]:
files,labels=images_and_labels(mo_data)

In [ ]:
from tqdm import tqdm

predicted=[]
for i in tqdm(range(len(files)), 'Predicting... '):
    predicted.append(image_to_pred(files[i],model))

## Top 4 predictions

In [ ]:
top_4_preds=[]
for i in range(len(predicted)):
    top_4_preds.append(top4_preds(predicted[i]))

In [ ]:
#using .softmax() function to convert ouptputs into probabilities of classification
m = nn.Softmax(dim=1)
softmax_preds=[]
for i in range(len(predicted)):
    softmax_preds.append(m(predicted[i]))

In [ ]:
#fills the list N_arr with all the false labels probability amount (populates no cat images)
N_arr=[]
amount=0
#filling false positive arr
for i in range(500):
    for j in range(10):
        if j in labels[i]:
            pass
            #donothing
        else:
            N_arr.append(softmax_preds[i][0][j].item())


In [ ]:
P_arr=[]
#fills the list P_arr with all the true labels probability amount (populates cat images)
for i in range(500):
    for j in range(4):
        correct_label_index = labels[i][j]
        P_arr.append(softmax_preds[i][0][correct_label_index].item())


In [ ]:
def num_corr(labels,predicted):
    tot_corr=[]
    for i in range(len(predicted)):
        num_corr=0
        for j in range(len(predicted[i])):
            if predicted[i][j] in labels[i]:
                num_corr+=1
        tot_corr.append(num_corr)
    return tot_corr

def total_num_of_pred(labels,predicted):
    tot_corr=[]
    total_num_of_corr = 0
    for i in range(len(predicted)):
        num_corr=0
        for j in range(len(predicted[i])):
            if predicted[i][j] in labels[i]:
                num_corr+=1
        tot_corr.append(num_corr)
        total_num_of_corr += num_corr
    return total_num_of_corr

In [ ]:
total=total_num_of_pred(labels,top_4_preds)
corr_per_file = num_corr(labels,top_4_preds)
len(top_4_preds)

In [ ]:
acc_eval=acc_table(labels,top_4_preds)

In [ ]:
for i in range(500):
  print('correct labels: ', labels[i], 'for image number ',i)
  print('top 4 predicted labels: ', top_4_preds[i])

In [ ]:
acc_eval

In [ ]:
acc_eval.head()

In [ ]:
plt.hist(acc_eval['Num_matches'])

In [ ]:
plot_correct_total(acc_eval,acc_eval)

In [ ]:
figure=plot_correct_total(acc_eval['Num_matches'],acc_eval,save=False,file_dest='corr_matches_500')

In [ ]:
plot_correct_prop(acc_eval['Num_matches'],acc_eval,save=False,file_dest='prop_corr_matches_500')

In [ ]:
ax1=pic_label_show(image=files[0],labels=labels[0],save=False,file_dest='SampleImage_actual')

In [ ]:
ax1=pic_label_show(image=files[1],labels=labels[1],save=False,file_dest='SampleImage_actual')

In [ ]:
ax1=pic_label_show(image=files[2],labels=labels[2],save=False,file_dest='SampleImage_actual')

In [ ]:
ax2=pic_label_show(image=files[0],labels=top_4_preds[0],predictions=True,save=False,file_dest='Sample_image_preds')

In [ ]:
ax2=pic_label_show(image=files[1],labels=top_4_preds[1],predictions=True,save=False,file_dest='Sample_image_preds')

In [ ]:
ax2=pic_label_show(image=files[2],labels=top_4_preds[2],predictions=True,save=False,file_dest='Sample_image_preds')

## Knee value calculations

In [ ]:
all_preds=[]
for i in range(len(predicted)):
    pred_array=predicted[i].numpy()
    temp=[]
    for j in range(pred_array.shape[1]):
        temp.append(pred_array[0][j])
    all_preds.append(temp)

In [ ]:
for i in range(len(all_preds)):
    all_preds[i]=sorted(all_preds[i],reverse=True)

In [ ]:
knee_val_no_assumption=gen_knees(all_preds)
knee_val_concave=gen_knees(all_preds,shape='concave')
knee_val_convex=gen_knees(all_preds,shape='convex')

In [ ]:
plot_knees(all_preds,knee_val=knee_val_no_assumption,save=False,file_dest='knee_val_500')

In [ ]:
init=np.zeros(25000)

In [ ]:
counts=[415,0,173,275,518,910,1577,5618,5176,13338]
index1=0
index2=0
for i in range(len(counts)):
    if i ==0:
        index2+=counts[i]
        init[index1:index2]=1
    else:
        index1+=counts[i-1]+1
        index2+=counts[i]+1
        init[index1:index2]=i+1

In [ ]:
print(str(np.mean(init)))
print(str(np.std(init)))

In [ ]:
plot_knees(all_preds,knee_val=knee_val_concave,shape='concave',save=False,file_dest='knee_val_500_concave')

In [ ]:
plot_knees(all_preds,knee_val=knee_val_convex,shape='convex',save=False,file_dest='knee_val_500_convex')

In [ ]:
############################################################################################
#BELOW IS CODE USED TO GET CDF WHICH IS USED TO GET THE TPR AND FPR COORDINATES TO PLOT ROC#
############################################################################################

In [ ]:
import math
import statistics as stat
#CDF formula calls for 1/2(1+ERF(x-mean/std*sqrt(2)))
#gets the fraction portion that needs to be ran through ERF
def get_x_erf(mean, std, x):
    res = 0
    num = x - mean
    den = std * math.sqrt(2)
    res = num / den
    return res

#math.erf() is called to give us a value that is then plugged into this which gives us the final CDF value
def cdf(erf):
    result = (1 + erf) * .5
    return result

#this function repopulates array 1 and 2 based on a new threshold of t
def new_t_val(arr1, arr2, t):
  #TRUE POSITIVES THOSE IN ARRAY 1 WITH A VALUE >= THE THRESHOLD
    newarr1=[]
  #FALSE POSITIVES THOSE IN ARRAY 2 WITH A VALUE >= THRESHOLD
    newarr2=[]
  #FALSE NEGATIVES THOSE IN ARRAY 1 WITH A VALUE < THRESHOLD
    newarr3=[]
  #TRUE NEGATIVES THOSE IN ARRAY 2 WITH A VALUE < THRESHOLD
    newarr4=[]
    '''
    for i in range(len(arr2)):
        newarr2.append(arr2[i])
    '''

    for i in range(len(arr1)):
        if arr1[i] < t:
            newarr3.append(arr1[i])
        else:
            newarr1.append(arr1[i])

    for i in range(len(arr2)):
        if arr2[i] < t:
            newarr4.append(arr2[i])
        else:
            newarr2.append(arr2[i])

    return newarr1,newarr2,newarr3,newarr4

#this function gets the coordinates of a TPR/FPR with the value of T increasing from .05 in intervals of .05
def getcoord(avg,std):
    coords=[]
    x = get_x_erf(avg,std,9999)
    erf = math.erf(x)
    cdf1 = cdf(erf)
    i = 0.05
    while i <= 1:
        y = get_x_erf(avg,std,i)
        erf2 = math.erf(y)
        cdf2 = cdf(erf2)
        coords.append(cdf1-cdf2)
        i += 0.05
    return coords
#gets the average of a list
def getavg(arr):
    total = 0
    for x in arr:
        total+=x
    average = total / len(arr)
    return average
#gets the STD of a list
def getstd(arr):
    x = stat.pstdev(arr)
    return x

#this function will take a TParr and FParr and go through the full process required to plot a ROC curve and finally plotting the ROC
#this function includes a parameter 'i' i is the new threshold to redefine arr1 and arr2
#this basically takes TParr and FParry and plots the ROC based on results all in one function
#but includes the option to repopulate array using
#new_t_val function see above for clarification
def method_1_through(arr1,arr2):
newarr1=[]
    newarr2=[]
    newarr3=[]
    newarr4=[]

    print('METHOD 1')
    print('arr1:',len(arr1))
    print('arr2:',len(arr2))

    avg1 = getavg(arr1)
    std1 = getstd(arr1)
    coord1 = getcoord(avg1,std1)
    avg2 = getavg(arr2)
    std2 = getstd(arr2)
    coord2 = getcoord(avg2,std2)
    print('Method 1 avg1=', avg1, 'Method 1 std1=', std1)
    print('Method 1 avg2=', avg2, 'Method 1 std2=', std2)
    print('TP-FP =', sum(coord1)-sum(coord2))
    #coord2 is FP on X-axis, coord1 is TP on Y-axis
    plt.scatter(coord2,coord1)
    plt.plot(coord2,coord1)
    plt.show()


def method_2_through(arr1,arr2,i):

    newarr1=[]
    newarr2=[]
    newarr3=[]
    newarr4=[]

    print('THRESHOLD OF:', i)
    print('arr1:',len(arr1))
    print('arr2:',len(arr2))
    newarr1, newarr2, newarr3, newarr4 = new_t_val(arr1,arr2,i)
    print('newarr1:',len(newarr1))
    print('newarr2:', len(newarr2))
    print('newarr3:',len(newarr3))
    print('newarr4:', len(newarr4))
    #newavg1 = getavg(newarr1)
    #newstd1 = getstd(newarr1)
    new_avg1 = getavg(newarr1+newarr2)
    newstd1 = getstd(newarr1+newarr2)
    coord1 = getcoord(newavg1,newstd1)
    #newavg2 = getavg(newarr2)
    #newstd2 = getstd(newarr2)
    newavg2 = getavg(newarr3+newarr4)
    newstd2 = getstd(newarr3+newarr4)
    coord2 = getcoord(newavg2,newstd2)
    print('newavg1=', newavg1, ' newstd1=', newstd1)
    print('newavg2=', newavg2, ' newstd2=', newstd2)
    print('TP-FP =', sum(coord1)-sum(coord2))
    #coord2 is FP on X-axis, coord1 is TP on Y-axis
    plt.scatter(coord2,coord1)
    plt.plot(coord2,coord1)
    plt.show()



In [ ]:
#populating a1
A1=[]
for i in range(len(P_arr)):
    A1.append(P_arr[i])


#populating a2
A2=[]
total2 = 0
for i in range(len(N_arr)):
    A2.append(N_arr[i])

In [ ]:
#getting total value for A1 and A2
total = 0
for i in range(2000):
    total += A1[i]

total1 = 0
for i in range(500):
    total1 += A2[i]

In [ ]:
newarr1 = A1
newarr2 = A2
print(len(A2))
print(len(A1))

In [ ]:
method_1_through(A1,A2)

In [ ]:
#Attempting to find the Top 2 values in the array
#import heapq
#top2 = heapq.nlargest(2,A1)
#i = min(top2)
#while i > 0:
#   method_2_through(A1,A2,i)
#    i=i-0.01

#Original Code
i = 0.01
while i <= 0.58:
  method_2_through(A1,A2,i)
  i=i+0.01

## Method 1 is Method 2 with threshold t set to 0: P_arr is TPRs and N_arr is FPRs

In [ ]:
method_2_through(A1,A2,0)
#ORIGINAL array without repopulating correct ROC CURVE

In [ ]:
newavg1 = getavg(A1)
newstd1 = getstd(A1)
coord1 = getcoord(newavg1,newstd1)
newavg2 = getstd(A2)
newstd2 = getstd(A2)
coord2 = getcoord(newavg2,newstd2)

In [ ]:
from kneed import KneeLocator, DataGenerator as dg
print('newavg1=', newavg1, ' newstd1=', newstd1)
print('newavg2=', newavg2, ' newstd2=', newstd2)
kl = KneeLocator(coord2, coord1, curve="concave")
kl.plot_knee()

In [ ]:
#method 3 , knee locator based on softmax prediction values

In [ ]:
w, h = 10, 500
arr = [[0 for x in range(w)] for y in range(h)]

In [ ]:
for i in range(500):
  for j in range(10):
    arr[i][j] = softmax_preds[i][0][j]

In [ ]:
for i in range(500):
  arr[i].sort()

xarr = [0,1,2,3,4,5,6,7,8,9]

In [ ]:
for i in range(500):
  kj = KneeLocator(xarr,arr[i], curve = "convex",direction="increasing")
  kj.plot_knee()

In [ ]:
#this function repopulates array 1 and 2 based on a new threshold of t
def updated_new_t_val(arr1, arr2, t):
  #TRUE POSITIVES THOSE IN ARRAY 1 WITH A VALUE >= THE THRESHOLD
    newarr1=[]
  #FALSE POSITIVES THOSE IN ARRAY 2 WITH A VALUE < THRESHOLD I
    newarr2=[]
  #FALSE NEGATIVES THOSE IN ARRAY 1 LESS THAN THRESHOLD I
    newarr3=[]
  #TRUE NEGATIVES THOSE IN ARRAY2 WITH A VALUE < THAN I
    newarr4=[]

    for i in range(len(arr2)):
        newarr2.append(arr2[i])

    for i in range(len(arr1)):
        if arr1[i] < t:
            newarr2.append(arr1[i])
        else:
            newarr1.append(arr1[i])

    return newarr1,newarr2